# Entrada para TPoint - Agrupamento sazonal dos mapas do SST
# Modelo de apontamento com termo FLOP

In [1]:
from astropy.time import Time
from astropy.coordinates import get_sun, AltAz
import numpy as np
import pandas as pd
from pandas_ods_reader import read_ods    # leitura de planilha do OpenOffice
import xlwt 
import CASLEO                             # coordenadas do observatorio 


In [2]:
# Leitura das datas/horas de equinocios e solsticios de 1999 a 2019
# ref: 
# https://pt.wikipedia.org/wiki/Equin%C3%B3cio

equinox_table = read_ods('eq_sol_1999-2019.ods')

# conversao para formato astropy.Time
VARGLOBAL_eq_sol_list = []
for index, row in equinox_table.iterrows():
    
    # Equinocio de marco
    eqmar = str(int(row['year'])).zfill(4)+ "-03-" +str(int(row['march_eq_day'])).zfill(2)+\
    "T"+row['march_eq_time']+":"+'00'
    
    # Solsticio de junho
    soljun = str(int(row['year'])).zfill(4)+ "-06-" +str(int(row['june_sol_day'])).zfill(2)+\
    "T"+row['june_sol_time']+":"+'00'
    
    # Equinocio de setembro
    eqset = str(int(row['year'])).zfill(4)+ "-09-" +str(int(row['sept_eq_day'])).zfill(2)+\
    "T"+row['sept_eq_time']+":"+'00'
    
    # Solsticio de dezembro
    soldec = str(int(row['year'])).zfill(4)+ "-12-" +str(int(row['dec_sol_day'])).zfill(2)+\
    "T"+row['dec_sol_time']+":"+'00'
    
    VARGLOBAL_eq_sol_list.append ([eqmar, soljun, eqset, soldec])

# Variavel de estacao do ano - INICIALIZADA COM VALOR FORA DO RANGE
estacao = 4

# Selecao de dados por estacao do ano
# usar eq_sol_list - tabela de equinocios e solsticios de 1999 a 2019 
# ref: 
# https://pt.wikipedia.org/wiki/Equin%C3%B3cio
# Linha da lista = ano
# coluna da lista = estacao; usar as constantes abaixo
MarchEq = 0
JuneSol = 1
SeptEq = 2
DecSol = 3
seasons = ['outono', 'inverno', 'primavera', 'verao']
date_str = ['03-20', '06-21', '09-23', '12-22']

# Variaveis de coeficientes do modelo de apontamento
# 30-jan-2022: Inclusão do coeficiente FLOP no modelo de apontamento
tp_IA = 0.0
tp_IE = [0.0, 0.0, 0.0, 0.0]
tp_NPAE = 0.0
tp_CA = [0.0, 0.0, 0.0, 0.0]
tp_AN = 0.0
tp_AW = 0.0
tp_FLOP = 0.0


In [3]:
##### FUNCOES PARA DEFINICAO DO NOME DO ARQUIVO DE SAIDA E DA STRING DO CABECALHO

def NomeArq_Estacao_Canal (prm_ano, prm_estacao, prm_canal, prm_iteracao):
    """
  
    Definicao do nome do arquivo de acordo com a estacao do ano e o canal.
    Retorna a string do nome do arquivo.
    
    """ 
    # A iteração inicial não muda, pois não tem modelo de apontamento aplicado
    if (prm_iteracao == 0):
        prefixo = 'indat_files/flop0/'
    elif (prm_iteracao == 1):
        prefixo =  'indat_files/flop1/'
    elif (prm_iteracao == 2):
        prefixo = 'indat_files/flop2/'
    else:
        pass

    if (prm_estacao == DecSol):
        ret_nome_arq = prefixo + str(prm_ano+1)+ '_' + seasons[prm_estacao] + '_ch' + str(prm_canal) + '.dat'
    elif (prm_estacao != 4):
        ret_nome_arq = prefixo + str(prm_ano)+ '_' + seasons[prm_estacao] + '_ch' + str(prm_canal) + '.dat'
    else:
        print ('******** Nao foi definida estacao do ano ********\n')
    return (ret_nome_arq)


def CabArq_Estacao_Canal (prm_ano, prm_estacao, prm_iteracao):
    """
    24-jan-2022: redução para apenas 1 iteração de refinamento, pois um segundo refinamento 
                 não produz redução significativa nos valores de RMS

    Definicao do cabecalho do arquivo de entrada pra o TPoint acordo com a estacao do ano e o canal.
    Retorna a string de cabeçalho.
    """

    # Indicacao do modelo aplicado 
    # prm_iteracao = 0 --> sem modelo
    # prm_iteracao = 1 --> modelo com FLOP aplicado

    if (prm_iteracao == 0):
        sufixo = ' - sem modelo aplicado'
    elif (prm_iteracao == 1):
        sufixo = ' - modelo inicial aplicado'
    elif (prm_iteracao == 2):
        sufixo = '- modelo da 1a iteracao aplicado'
    else: sufixo = ' '
        
    if (prm_estacao == DecSol):
        ret_periodo = 'Mapas - '+ seasons[prm_estacao] + ' ' + str(prm_ano) + '-' + str(prm_ano+1) + sufixo
    elif (estacao != 4):
        ret_periodo = 'Mapas - '+ seasons[prm_estacao] + ' ' + str(prm_ano) + sufixo
    else:
        print ('******** Nao foi definida estacao do ano ********\n')
    return (ret_periodo)


def NomeArq_outmex_Estacao_Canal (prm_ano, prm_estacao, prm_canal, prm_iteracao):
    """
    24-jan-2022: redução para apenas 1 iteração de refinamento, pois um segundo refinamento 
                 não produz redução significativa nos valores de RMS

    Definicao do nome do arquivo do modelo exportado (formato Excel) de acordo com a estacao do ano e o canal.
    Retorna a string do nome do arquivo.
    """ 
    if (prm_iteracao == 0):
        prefixo = 'outmex_files/flop0/tpmex_'
    elif (prm_iteracao == 1):
        prefixo =  'outmex_files/flop1/tpmex_'
    elif (prm_iteracao == 2):
        prefixo = 'outmex_files/flop2/tpmex_'
    else: prefixo = ' '

    if (prm_estacao == DecSol):
        ret_nome_arq = prefixo + str(prm_ano+1)+ '_' + seasons[prm_estacao] + '_ch' + str(prm_canal) + '.dat'
    elif (prm_estacao != 4):
        ret_nome_arq = prefixo + str(prm_ano)+ '_' + seasons[prm_estacao] + '_ch' + str(prm_canal) + '.dat'
    else:
        print ('******** Nao foi definida estacao do ano ********\n')
    return (ret_nome_arq)


In [4]:
# LEITURA DO ARQUIVO DE DADOS .ODS (dados dos canais separados por planilha)
# segundo argumento de read_ods: canal de 212 GHz a ser analisado

def Ler_Dados_Estacao_Canal (prm_ano, prm_estacao, prm_canal):
    """
    Leitura do arquivo de dados .ods e filtragem dos dados de interesse
    Uso de VARGLOBAL_eq_sol_list (datas de equinocio e solsticio de 1999 a 2019)
    Retorna o dataframe com os dados de interesse
    """
    data_ch = read_ods('centro_raio_212.ods', prm_canal)
    
    dados_interesse = []

    # data_ch_of_interest deverá conter apenas os dados de interesse para ajuste do modelo do TPoint

    # Inicio e termino da estacao de interesse 
    # constantes de equinocio e solsticio = colunas de eq_sol_list
    datahora_ini = Time(VARGLOBAL_eq_sol_list[prm_ano-1999][prm_estacao], format='isot', scale='utc')

    if (estacao == DecSol):
        datahora_fim = Time(VARGLOBAL_eq_sol_list[1+prm_ano-1999][MarchEq], format='isot', scale='utc')
    elif (estacao != 4):
        datahora_fim = Time(VARGLOBAL_eq_sol_list[prm_ano-1999][prm_estacao+1], format='isot', scale='utc')
    else:
        print ('******** Nao foi definida estacao do ano ********\n')

    # Filtrar somente os dados da estacao selecionada
    for index, row in data_ch.iterrows():
    
        ## data e hora do mapa
        # montagem da string de data no formato Time : 'YYYY-MO-DATHH:MM:SS'
        dtstamp_mapa = str(int(row['YEAR'])).zfill(4)+"-"+\
        str(int(row['MO'])).zfill(2)+"-"+\
        str(int(row['DA'])).zfill(2)+\
        "T"+str(int(row['HR'])).zfill(2)+":"+\
        str(int(row['MI'])).zfill(2)+":"+'00'
        
        datahora_mapa = Time(dtstamp_mapa, format='isot', scale='utc') 

        if (datahora_mapa >= datahora_ini) and (datahora_mapa < datahora_fim):
            # acumular na lista os dados pertencentes à estação de interesse, com todas as colunas que serão usadas
            dados_interesse.append ([row['YEAR'], row['MO'], row['DA'], row['HR'],\
                                     row['MI'], row['X_cntr'], row['Y_cntr']])

    # gerar o dataframe
    data_ch_of_interest = pd.DataFrame (dados_interesse, columns=['YEAR','MO','DA','HR','MI','X_cntr','Y_cntr'])
    
    return data_ch_of_interest


In [5]:
##### LEITURA DE ARQUIVO OUTMEX DE MODELO DE APONTAMENTO

def Le_Modelo_outmex_FLOP (prm_ano, prm_estacao, prm_iteracao):
    """
    30 Jan 2022: 
        Inclusão do coeficiente FLOP no modelo de apontamento
    
    Le o arquivo do modelo para o ano "prm_ano" e a estação "prm_estacao".
    Modelos iniciais (prm_iteracao = 0) ficam na subpasta 'outmod_files/flop0'.
    Modelos da 1a iteracao (prm_iteracao = 1) ficam na subpasta 'outmod_files/flop1'.
    Modelos da 2a iteracao (prm_iteracao = 2) ficam na subpasta 'outmod_files/flop2'.
    
    Retorna os coeficientes do modelo de apontamento do SST
    * IA
    * IE (um valor para cada corneta)
    * NPAE
    * CA (um valor para cada corneta)
    * AN
    * AW 
    * FLOP
    
    Formato do arquivo:
    SST|T|qtd_medidas|EW_RMS|NS_RMS|LR_RMS|UD_RMS|sky_RMS|refaction_a|refaction_b|\
    <spaces>|<spaces>|"IA"|valor IA|rms IA|\
    <spaces>|"IE/1"|valor IE1|rms IE1|\
    <spaces>|"IE/2"|valor IE2|rms IE2|\
    <spaces>|"IE/3"|valor IE3|rms IE3|\
    <spaces>|"IE/4"|valor IE4|rms IE4|\
    <spaces>|"NPAE"|valor NPAE|rms NPAE|\
    <spaces>|"CA/1"|valor CA1|rms CA1|\
    <spaces>|"CA/2"|valor CA2|rms CA2|\
    <spaces>|"CA/3"|valor CA3|rms CA3|\
    <spaces>|"CA/4"|valor CA4|rms CA4|\
    <spaces>|"AN"|valor AW|rms AW|\
    <spaces>|"AW"|valor AN|rms AN|
    <spaces>|"FLOP"|valor FLOP|rms FLOP|
    
    Conteudo de 'linha_dados_clean':
    [0]  'SST'
    [1]  'T'
    [2]  número de observações ativas
    [3]  EW_RMS
    [4]  NS_RMS
    [5]  LR_RMS
    [6]  UD_RMS
    [7]  SKY_RMS
    [8]  constante de refração a
    [9]  constante de refração b
    [10] 'IA'
    [11] valor de IA
    [12] rms de IA
    [13] 'IE/1'
    [14] valor de IE do canal 1
    [15] rms de IE do canal 1
    [16] 'IE/2'
    [17] valor de IE do canal 2
    [18] rms de IE do canal 2
    [19] 'IE/3'
    [20] valor de IE do canal 3
    [21] rms de IE do canal 3
    [22] 'IE/4'
    [23] valor de IE do canal 4
    [24] rms de IE do canal 4
    [25] 'NPAE'
    [26] valor de NPAE
    [27] rms de NPAE
    [28] 'CA/1'
    [29] valor de CA do canal 1
    [30] rms de CA do canal 1
    [31] 'CA/2'
    [32] valor de CA do canal 2
    [33] rms de CA do canal 2
    [34] 'CA/3'
    [35] valor de CA do canal 3
    [36] rms de CA do canal 3
    [37] 'CA/4'
    [38] valor de CA do canal 4
    [39] rms de CA do canal 4
    [40] 'AN'
    [41] valor de AN
    [42] rms de AN
    [43] 'AW'
    [44] valor de AW
    [45] rms de AW
    [46] 'FLOP'
    [47] valor de FLOP
    [48] rms de FLOP
    """

    # variaveis locais de coeficientes
    local_tp_IA = 0.0
    local_tp_IE = [0.0, 0.0, 0.0, 0.0]
    local_tp_NPAE = 0.0
    local_tp_CA = [0.0, 0.0, 0.0, 0.0]
    local_tp_AN = 0.0
    local_tp_AW = 0.0
    local_tp_FLOP = 0.0

    if (prm_iteracao != 0):
        # Nome do arquivo do modelo de apontamento
        if (prm_estacao == DecSol):
            ano_mod = 1+prm_ano
        else:
            ano_mod = prm_ano

        # Nome do arquivo do modelo de apontamento
        if (prm_estacao == DecSol):
            ano_mod = 1+prm_ano
        else:
            ano_mod = prm_ano

        # String do nome do arquivo
        # depende do modelo aplicado
        if (prm_iteracao == 1):
            nomearq_mod = 'outmex_files/flop0/tpmex_FLOP_' + str(ano_mod) + '_' + seasons[prm_estacao] + '.dat'
        elif prm_iteracao == 2:
            nomearq_mod = 'outmex_files/flop1/tpmex_FLOP_' + str(ano_mod) + '_' + seasons[prm_estacao] + '.dat'
        print (nomearq_mod)

        # Leitura do arquivo com 'limpeza' dos espaços em branco das linhas
        # resulta em uma lista de linhas - por sua vez, cada linha é uma lista de strings
        infile = open(nomearq_mod, "r")
        linha_dados = next(infile)
        infile.close()

        # conteudo da linha 
        linha_dados_clean = linha_dados.replace("|", " ")
        lista_dados = linha_dados_clean.split()

        ## echo print do modelo
        print ('MODELO EM', nomearq_mod, '(segundos de arco)')
        print ('IA   = ', lista_dados[11])
        print ('IE   = ', lista_dados[14], lista_dados[17], lista_dados[20], lista_dados[23])
        print ('NPAE = ', lista_dados[26])
        print ('CA   = ', lista_dados[29], lista_dados[32], lista_dados[35], lista_dados[38])
        print ('AN   = ', lista_dados[41])
        print ('AW   = ', lista_dados[44])
        print ('FLOP = ', lista_dados[47], '\n')

        # OBS: arquivo de modelo do TPoint em segundos de arco
        # dividir por 3600 para obter graus decimais
        local_tp_IA = float(lista_dados[11])/3600
        local_tp_IE[0] = float(lista_dados[14])/3600
        local_tp_IE[1] = float(lista_dados[17])/3600
        local_tp_IE[2] = float(lista_dados[20])/3600
        local_tp_IE[3] = float(lista_dados[23])/3600
        local_tp_NPAE = float(lista_dados[26])/3600
        local_tp_CA[0] = float(lista_dados[29])/3600
        local_tp_CA[1] = float(lista_dados[32])/3600
        local_tp_CA[2] = float(lista_dados[35])/3600
        local_tp_CA[3] = float(lista_dados[38])/3600
        local_tp_AN = float(lista_dados[41])/3600
        local_tp_AW = float(lista_dados[44])/3600
        local_tp_FLOP = float(lista_dados[47])/3600
   
    print ('Modelo de apontamento (graus decimais):')
    print ('IA:   ', local_tp_IA)
    print ('IE:   ', local_tp_IE)
    print ('NPAE: ', local_tp_NPAE)
    print ('CA:   ', local_tp_CA) 
    print ('AN:   ', local_tp_AN)
    print ('AW:   ', local_tp_AW)
    print ('FLOP: ', local_tp_FLOP)
    return (local_tp_IA, local_tp_IE, local_tp_NPAE, local_tp_CA, local_tp_AN, local_tp_AW, local_tp_FLOP)


In [6]:
def Calcula_Dados_TPoint_Estacao_Canal (prm_data_ch_of_interest):
    """
    30 Jan 2022: 
        (1) Inclusão do coeficiente FLOP no modelo de apontamento

    Iteracao em prm_data_ch_of_interest para gerar os dados para o TPoint
    Retorna o dataframe com os dados a serem gravados
    """

    # criacao da lista de saida
    entradas_TPoint = []

    # colunas a serem rotuladas na geracao do arquivo:  
    # 'AzReal': float, graus decimais
    # 'ElReal': float, graus decimais
    # 'AzMount': float, graus decimais
    # 'ElMount': float, graus decimais
    # 'DataHora': 'YYYY-MO-DA HH:MM:SS' - comentario no final da linha

    # geracao dos registros de dados:
    #
    # EphAz-OffsetAz    EphEl-OffsetEl    EphAz+PM_DeltaAz    EphEl+PM_DeltaEl
    # 21 caracteres por coluna, preenchidos com espaços em branco, alinhamento à esquerda
    #
    # Onde:    Eph - efemerides do Sol na data/hora do mapa
    #          OffsetAz, OffsetEl - dados calculados pelo Fabian, arquivo centro_raio_212.ods
    #          PM_DeltaAz, PM_DeltaEl - modelo de apontamento do TPoint na data/hora de levantamento do mapa 

    for index, row in prm_data_ch_of_interest.iterrows():
        
        ### montagem da string de data no formato exigido por get_sun: 'YYYY-MO-DA HH:MM:SS'
        datahora = str(int(row['YEAR'])).zfill(4)+"-"+\
        str(int(row['MO'])).zfill(2)+"-"+\
        str(int(row['DA'])).zfill(2)+" "+\
        str(int(row['HR'])).zfill(2)+":"+\
        str(int(row['MI'])).zfill(2)+":"+'00'

        ### calculo das efemerides em coordenadas equatoriais
        # resultado: (ascensao reta em graus, declinacao em graus, distancia Terra-Sol em UA)
        posicao_sol = get_sun (Time(datahora))
    
        ### conversao para coordenadas horizontais do SST
        # resultado: (az em graus, el em graus, distancia Terra-Sol em metros)
        # formato das saidas de azimute e elevacao: (S)xxdyymzz.zzzzs
        posicao_sol_az = posicao_sol.transform_to (AltAz (location=CASLEO.Observatory_Coordinates()))

        ### calculo do angulo decimal 
        # conversao para string para 'fatiar' mais facilmente 
        # (deselegante, mas funciona) 

        # azimute
        string_az = str(posicao_sol_az.az) 
        posicao_d = string_az.find('d')  # fim do segmento do grau
        posicao_m = string_az.find('m')  # fim do segmento do minuto
        posicao_s = string_az.find('s')  # fim do segmento do segundo
        valor_grau = float (string_az[0:posicao_d])
        valor_minuto = float (string_az[posicao_d+1:posicao_m]) 
        valor_segundo = float (string_az[posicao_m+1:posicao_s])
        angulo_dec_az = float (valor_grau) + float(valor_minuto) / 60 + float(valor_segundo) / 3600

        # elevacao
        string_elev = str(posicao_sol_az.alt) 
        posicao_d = string_elev.find('d')  # fim do segmento do grau
        posicao_m = string_elev.find('m')  # fim do segmento do minuto
        posicao_s = string_elev.find('s')  # fim do segmento do segundo
        valor_grau = float (string_elev[0:posicao_d])
        valor_minuto = float (string_elev[posicao_d+1:posicao_m]) 
        valor_segundo = float (string_elev[posicao_m+1:posicao_s])
        angulo_dec_elev = float (valor_grau) + float(valor_minuto) / 60 + float(valor_segundo) / 3600

        ### Calculo da posicao real do centro solar 
        # Coordenadas do centro do disco em segundos de arco; faz-se a conversao para graus decimais
        AzReal = angulo_dec_az - (float(row['X_cntr'])/3600)
        ElReal = angulo_dec_elev - (float(row['Y_cntr'])/3600)

        ### Calculo das posicoes comandadas aos encoders
        # DEPENDE DO MODELO USADO NO TPOINT
        # CADA CONJUNTO DE PARÂMETROS TEM SEU CÁLCULO
        
        # Conversao das efemerides para radianos
        angulo_rad_az = np.radians (angulo_dec_az)
        angulo_rad_elev = np.radians (angulo_dec_elev)

        ### ALTERAR AQUI PARA CADA MOD. APONTAMENTO ADOTADO 
        # desvio em Azimute
        deltaAz_AN = -1*tp_AN*np.sin(angulo_rad_az)*np.tan(angulo_rad_elev)
        deltaAz_AW = -1*tp_AW*np.cos(angulo_rad_az)*np.tan(angulo_rad_elev)
        deltaAz_CA = -1*tp_CA[canal_analise-1]/np.cos(angulo_rad_elev)
        deltaAz_IA = -1*tp_IA
        deltaAz_NPAE = -1*tp_NPAE*np.tan(angulo_rad_elev)
        deltaAz_modelo = deltaAz_AN + deltaAz_AW + deltaAz_CA + deltaAz_IA + deltaAz_NPAE

        # desvio em Elevacao
        deltaEl_AN = -1*tp_AN*np.cos(angulo_rad_az)
        deltaEl_AW = tp_AW*np.sin(angulo_rad_az)
        deltaEl_IE = tp_IE[canal_analise-1]
        deltaEl_FLOP = tp_FLOP
        deltaEl_modelo = deltaEl_AN + deltaEl_AW + deltaEl_IE + deltaEl_FLOP

        ### FIM DO TRECHO DEPENDENTE DO MOD. APONTAMENTO

        # posicoes dos encoders 
        AzMount = angulo_dec_az + deltaAz_modelo
        ElMount = angulo_dec_elev + deltaEl_modelo

        # Registro dos dados na lista de saida
        entradas_TPoint.append ([AzReal, ElReal, AzMount, ElMount, datahora, deltaAz_modelo, deltaEl_modelo])

    # Atribuicao da lista a um Data Frame com rotulacao das colunas
    df_entradas_TPoint = pd.DataFrame (entradas_TPoint, columns=['AzReal','ElReal','AzMount','ElMount','DataHora','DeltaAz','DeltaEl'])

    return (df_entradas_TPoint)


In [7]:
def Grava_Dados_TPoint_Estacao_Canal (prm_nomearq, prm_periodo, prm_canal, prm_iteracao, dados_TPoint):
    """
    Gravacao de dados_TPoint em arquivo 
    """    

    arqsaida=open(prm_nomearq,mode="w",encoding="utf-8")

    # cabecalho do arquivo
    arqsaida.write('!\n')
    if (prm_iteracao == 0):
        arqsaida.write('! Az/El do centro do Sol, calculado a partir dos dados do Fabian\n') 
    elif (prm_iteracao == 1):
        arqsaida.write('! Az/El do centro do Sol com correcao do modelo com FLOP\n') 
    else:
        arqsaida.write('! Az/El do centro do Sol com correcao do modelo com FLOP da 1a iteracao\n') 

    arqsaida.write('!\n\n')
    arqsaida.write('SST\n')
    arqsaida.write(': NODA\n')
    arqsaida.write(': ALTAZ\n')
    arqsaida.write(': SST\n')
    arqsaida.write('-31 47 56.3\n\n')
    arqsaida.write('! Formato 4 do TPoint\n')
    arqsaida.write('! ')
    arqsaida.write(str(prm_periodo))
    arqsaida.write('\n')
    arqsaida.write('! Canal ')
    arqsaida.write(str(prm_canal))
    arqsaida.write('\n')
    arqsaida.write('!    Az real     El real   Az encoder   El encoder ! Data Hora (UTC)         DltAzMod    DltElMod\n')
    arqsaida.write('! -------------------------------------------------------------------------------------------------\n\n')

    for index, row in dados_TPoint.iterrows():
        # limpar o buffer
        buffer_str = ''

        # colunas numericas
        buffer_str += str(round(row['AzReal'], 4)).rjust(12)
        buffer_str += str(round(row['ElReal'], 4)).rjust(12)
        buffer_str += str(round(row['AzMount'], 4)).rjust(12)
        buffer_str += str(round(row['ElMount'], 4)).rjust(12)
        buffer_str += '   ! '
        buffer_str += row['DataHora'] 
        buffer_str += str(round(row['DeltaAz'], 4)).rjust(12)
        buffer_str += str(round(row['DeltaEl'], 4)).rjust(12)
        buffer_str += '\n' 
        arqsaida.write(buffer_str)

    arqsaida.close()
    
    return


In [8]:
def Le_RMS_outmex_file (prm_ano, prm_estacao, prm_iteracao):
    """
    30 Jan 2022: 
        (1) Inclusão do coeficiente FLOP no modelo de apontamento

    Le o arquivo do modelo para o ano "prm_ano" e a estação "prm_estacao".
    Modelos iniciais (prm_iteracao = 0) ficam na subpasta 'outmod_files/flop0'.
    Modelos da 1a iteracao (prm_iteracao = 1) ficam na subpasta 'outmod_files/flop1'.
    
    Retorna uma string de data para Excel/LibreOffice e os valores de RMS do modelo de apontamento
    
    Formato do arquivo:
    SST|T|qtd_medidas|EW_RMS|NS_RMS|LR_RMS|UD_RMS|sky_RMS|refaction_a|refaction_b|\
    <spaces>|<spaces>|"IA"|valor IA|rms IA|\
    <spaces>|"IE/1"|valor IE1|rms IE1|\
    <spaces>|"IE/2"|valor IE2|rms IE2|\
    <spaces>|"IE/3"|valor IE3|rms IE3|\
    <spaces>|"IE/4"|valor IE4|rms IE4|\
    <spaces>|"NPAE"|valor NPAE|rms NPAE|\
    <spaces>|"CA/1"|valor CA1|rms CA1|\
    <spaces>|"CA/2"|valor CA2|rms CA2|\
    <spaces>|"CA/3"|valor CA3|rms CA3|\
    <spaces>|"CA/4"|valor CA4|rms CA4|\
    <spaces>|"AN"|valor AW|rms AW|\
    <spaces>|"AW"|valor AN|rms AN|
    <spaces>|"FLOP"|valor FLOP|rms FLOP|
    
    Conteudo de 'linha_dados_clean':
    [0]  'SST'
    [1]  'T'
    [2]  número de observações ativas
    [3]  EW_RMS
    [4]  NS_RMS
    [5]  LR_RMS
    [6]  UD_RMS
    [7]  SKY_RMS
    [8]  constante de refração a
    [9]  constante de refração b
    [10] 'IA'
    [11] valor de IA
    [12] rms de IA
    [13] 'IE/1'
    [14] valor de IE do canal 1
    [15] rms de IE do canal 1
    [16] 'IE/2'
    [17] valor de IE do canal 2
    [18] rms de IE do canal 2
    [19] 'IE/3'
    [20] valor de IE do canal 3
    [21] rms de IE do canal 3
    [22] 'IE/4'
    [23] valor de IE do canal 4
    [24] rms de IE do canal 4
    [25] 'NPAE'
    [26] valor de NPAE
    [27] rms de NPAE
    [28] 'CA/1'
    [29] valor de CA do canal 1
    [30] rms de CA do canal 1
    [31] 'CA/2'
    [32] valor de CA do canal 2
    [33] rms de CA do canal 2
    [34] 'CA/3'
    [35] valor de CA do canal 3
    [36] rms de CA do canal 3
    [37] 'CA/4'
    [38] valor de CA do canal 4
    [39] rms de CA do canal 4
    [40] 'AN'
    [41] valor de AN
    [42] rms de AN
    [43] 'AW'
    [44] valor de AW
    [45] rms de AW
    [46] 'FLOP'
    [47] valor de FLOP
    [48] rms de FLOP
    """

    # Nome do arquivo do modelo de apontamento
    if (prm_estacao == DecSol):
        ano_mod = 1+prm_ano
    else:
        ano_mod = prm_ano

    # String do nome do arquivo
    # depende do modelo aplicado
    if (prm_iteracao == 0):
        nomearq_mod = 'outmex_files/flop0/tpmex_FLOP_' + str(ano_mod) + '_' + seasons[prm_estacao] + '.dat'
    elif prm_iteracao == 1:
        nomearq_mod = 'outmex_files/flop1/tpmex_FLOP_' + str(ano_mod) + '_' + seasons[prm_estacao] + '.dat'
    else: 
        nomearq_mod = 'outmex_files/flop2/tpmex_FLOP_' + str(ano_mod) + '_' + seasons[prm_estacao] + '.dat'

    # Leitura do arquivo com 'limpeza' dos espaços em branco das linhas
    # resulta em uma lista de linhas - por sua vez, cada linha é uma lista de strings
    infile = open(nomearq_mod, "r")
    linha_dados = next(infile)
    infile.close()

    # conteudo da linha 
    linha_dados_clean = linha_dados.replace("|", " ")
    lista_dados = linha_dados_clean.split()
    return (str(prm_ano) + '-' + date_str[prm_estacao], 
            float(lista_dados[3]),\
            float(lista_dados[4]),\
            float(lista_dados[5]),\
            float(lista_dados[6]),\
            float(lista_dados[7]))


In [9]:
def Le_termos_outmex_file (prm_ano, prm_estacao, prm_iteracao):
    """
    Le o arquivo do modelo para o ano "prm_ano" e a estação "prm_estacao".
    Modelos iniciais (prm_iteracao = 0) ficam na subpasta 'outmod_files/flop0'.
    Modelos da 1a iteracao (prm_iteracao = 1) ficam na subpasta 'outmod_files/flop1'.
    
    Retorna uma string de data para Excel/OpenOffice e os valores dos termos do modelo de apontamento
    
    Formato do arquivo:
    SST|T|qtd_medidas|EW_RMS|NS_RMS|LR_RMS|UD_RMS|sky_RMS|refaction_a|refaction_b|\
    <spaces>|<spaces>|"IA"|valor IA|rms IA|\
    <spaces>|"IE/1"|valor IE1|rms IE1|\
    <spaces>|"IE/2"|valor IE2|rms IE2|\
    <spaces>|"IE/3"|valor IE3|rms IE3|\
    <spaces>|"IE/4"|valor IE4|rms IE4|\
    <spaces>|"NPAE"|valor NPAE|rms NPAE|\
    <spaces>|"CA/1"|valor CA1|rms CA1|\
    <spaces>|"CA/2"|valor CA2|rms CA2|\
    <spaces>|"CA/3"|valor CA3|rms CA3|\
    <spaces>|"CA/4"|valor CA4|rms CA4|\
    <spaces>|"AN"|valor AN|rms AN|\
    <spaces>|"AW"|valor AW|rms AW|\
    <spaces>|"FLOP"|valor FLOP|rms FLOP|
    
    Conteudo de 'linha_dados_clean':
    [0]  'SST'
    [1]  'T'
    [2]  número de observações ativas
    [3]  EW_RMS
    [4]  NS_RMS
    [5]  LR_RMS
    [6]  UD_RMS
    [7]  SKY_RMS
    [8]  constante de refração a
    [9]  constante de refração b
    [10] 'IA'
    [11] valor de IA
    [12] rms de IA
    [13] 'IE/1'
    [14] valor de IE do canal 1
    [15] rms de IE do canal 1
    [16] 'IE/2'
    [17] valor de IE do canal 2
    [18] rms de IE do canal 2
    [19] 'IE/3'
    [20] valor de IE do canal 3
    [21] rms de IE do canal 3
    [22] 'IE/4'
    [23] valor de IE do canal 4
    [24] rms de IE do canal 4
    [25] 'NPAE'
    [26] valor de NPAE
    [27] rms de NPAE
    [28] 'CA/1'
    [29] valor de CA do canal 1
    [30] rms de CA do canal 1
    [31] 'CA/2'
    [32] valor de CA do canal 2
    [33] rms de CA do canal 2
    [34] 'CA/3'
    [35] valor de CA do canal 3
    [36] rms de CA do canal 3
    [37] 'CA/4'
    [38] valor de CA do canal 4
    [39] rms de CA do canal 4
    [40] 'AN'
    [41] valor de AN
    [42] rms de AN
    [43] 'AW'
    [44] valor de AW
    [45] rms de AW
    [46] 'FLOP'
    [47] valor de FLOP
    [48] rms de FLOP
    """

    # Nome do arquivo do modelo de apontamento
    if (prm_estacao == DecSol):
        ano_mod = 1+prm_ano
    else:
        ano_mod = prm_ano

    # String do nome do arquivo
    # depende do modelo aplicado
    if (prm_iteracao == 0):
        nomearq_mod = 'outmex_files/flop0/tpmex_FLOP_' + str(ano_mod) + '_' + seasons[prm_estacao] + '.dat'
    elif prm_iteracao == 1:
        nomearq_mod = 'outmex_files/flop1/tpmex_FLOP_' + str(ano_mod) + '_' + seasons[prm_estacao] + '.dat'
    else: 
        nomearq_mod = 'outmex_files/refina2/tpmex_FLOP_' + str(ano_mod) + '_' + seasons[prm_estacao] + '.dat'

    # Leitura do arquivo com 'limpeza' dos espaços em branco das linhas
    # resulta em uma lista de linhas - por sua vez, cada linha é uma lista de strings
    infile = open(nomearq_mod, "r")
    linha_dados = next(infile)
    infile.close()

    # conteudo da linha 
    linha_dados_clean = linha_dados.replace("|", " ")
    lista_dados = linha_dados_clean.split()
    return (str(prm_ano) + '-' + date_str[prm_estacao], 
            float(lista_dados[11]),\
            float(lista_dados[14]),\
            float(lista_dados[17]),\
            float(lista_dados[20]),\
            float(lista_dados[23]),\
            float(lista_dados[26]),\
            float(lista_dados[29]),\
            float(lista_dados[32]),\
            float(lista_dados[35]),\
            float(lista_dados[38]),\
            float(lista_dados[41]),\
            float(lista_dados[44]),\
            float(lista_dados[47]))


### ROTINA PRINCIPAL - CONSOLIDAÇÃO DE MODELOS EM UMA PLANILHA ###
#   30 Jan 2022: 
#       (1) Inclusão do coeficiente FLOP no modelo de apontamento
#       (2) Redução para apenas 1 iteração de refinamento, pois um segundo refinamento não
#           produz redução significativa nos valores de RMS

# NOTA 0: Para usar, mudar a célula de "Markdown" para "Code"

# Pular estações sem modelos calculados:
#     MarchEq 1999 - outono 1999
#     SeptEq 2000 - primavera 2000
#     SeptEq 2005 - primavera 2005
#     DecSol 2005 - verao 2006
#     DecSol 2016 - verao 2017

lista_rms = [0, 0, 0, 0, 0, 0]
saida_rms = []

prm_iteracao = 0

# exceções ao loop:
# 1999 - verao
# 1999 - outono 
# 2000 - primavera
# primavera 2005 - SeptEq 2005
# verao 2006 - DecSol 2005
# verao 2017 - DecSol 2016

for prm_ano in range (2007, 2017):
    for prm_estacao in range (0,4):
        if (prm_ano == 1999 and prm_estacao == MarchEq): 
            pass
        elif (prm_ano == 2000 and prm_estacao == SeptEq):
            pass
        elif (prm_ano == 2005 and prm_estacao == SeptEq):
            pass
        elif (prm_ano == 2005 and prm_estacao == DecSol):
            pass
        elif (prm_ano == 2016 and prm_estacao == DecSol):
            pass
        elif (prm_ano == 2019 and prm_estacao == DecSol):
            pass
        else:
            saida_rms.append (Le_RMS_outmex_file (prm_ano, prm_estacao, prm_iteracao))

df_saida_rms_init = pd.DataFrame (saida_rms, columns= ['TIMESTAMP', 'EW_RMS', 'NS_RMS', 'LR_RMS', 'UD_RMS', 'sky_RMS'])
print (df_saida_rms_init)

# primeiro refinamento
lista_rms = [0, 0, 0, 0, 0, 0]
saida_rms = []
prm_iteracao = 1 

for prm_ano in range (2007, 2017):
    for prm_estacao in range (0,4):
        if (prm_ano == 1999 and prm_estacao == MarchEq): 
            pass
        elif (prm_ano == 2000 and prm_estacao == SeptEq):
            pass
        elif (prm_ano == 2005 and prm_estacao == SeptEq):
            pass
        elif (prm_ano == 2005 and prm_estacao == DecSol):
            pass
        elif (prm_ano == 2016 and prm_estacao == DecSol):
            pass
        elif (prm_ano == 2019 and prm_estacao == DecSol):
            pass
        else:
            saida_rms.append (Le_RMS_outmex_file (prm_ano, prm_estacao, prm_iteracao))

df_saida_rms_refina1 = pd.DataFrame (saida_rms, columns= ['TIMESTAMP', 'EW_RMS', 'NS_RMS', 'LR_RMS', 'UD_RMS', 'sky_RMS'])
print (df_saida_rms_refina1)

# segundo refinamento
#lista_rms = [0, 0, 0, 0, 0, 0]
#saida_rms = []
#prm_iteracao = 2 

#for prm_ano in range (1999, 2017):
#    for prm_estacao in range (0,4):
#        if (prm_ano == 1999 and prm_estacao == MarchEq): 
#            pass
#        elif (prm_ano == 2000 and prm_estacao == SeptEq):
#            pass
#        elif (prm_ano == 2005 and prm_estacao == SeptEq):
#            pass
#        elif (prm_ano == 2005 and prm_estacao == DecSol):
#            pass
#        elif (prm_ano == 2016 and prm_estacao == DecSol):
#            pass
#        elif (prm_ano == 2019 and prm_estacao == DecSol):
#            pass
#        else:
#            saida_rms.append (Le_RMS_outmex_file (prm_ano, prm_estacao, prm_iteracao))

#df_saida_rms_refina2 = pd.DataFrame (saida_rms, columns= ['TIMESTAMP', 'EW_RMS', 'NS_RMS', 'LR_RMS', 'UD_RMS', 'sky_RMS'])
#print (df_saida_rms_refina2)

df_saida_rms_init.to_excel('TP_RMS_values_flop0.xls')
df_saida_rms_refina1.to_excel('TP_RMS_values_flop1.xls')


In [10]:
### ROTINA PRINCIPAL - CONSOLIDAÇÃO DE MODELOS EM UMA PLANILHA ###
### TERMOS DOS MODELOS ###

#   08 Set 2022: 
#       (*) Rotina para consolidar termos dos modelos de apontamento
#   30 Jan 2022: 
#       (1) Inclusão do coeficiente FLOP no modelo de apontamento
#       (2) Redução para apenas 1 iteração de refinamento, pois um segundo refinamento não
#           produz redução significativa nos valores de RMS

# NOTA 0: Para usar, mudar a célula de "Markdown" para "Code"

# Pular estações sem modelos calculados:
#     MarchEq 1999 - outono 1999
#     SeptEq 2000 - primavera 2000
#     SeptEq 2005 - primavera 2005
#     DecSol 2005 - verao 2006
#     DecSol 2016 - verao 2017

lista_termos = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
saida_termos = []

prm_iteracao = 0

# exceções ao loop:
# 1999 - verao
# 1999 - outono 
# 2000 - primavera
# primavera 2005 - SeptEq 2005
# verao 2006 - DecSol 2005
# verao 2017 - DecSol 2016

for prm_ano in range (2007, 2017):
    for prm_estacao in range (0,4):
        if (prm_ano == 1999 and prm_estacao == MarchEq): 
            pass
        elif (prm_ano == 2000 and prm_estacao == SeptEq):
            pass
        elif (prm_ano == 2005 and prm_estacao == SeptEq):
            pass
        elif (prm_ano == 2005 and prm_estacao == DecSol):
            pass
        elif (prm_ano == 2016 and prm_estacao == DecSol):
            pass
        elif (prm_ano == 2019 and prm_estacao == DecSol):
            pass
        else:
            saida_termos.append (Le_termos_outmex_file (prm_ano, prm_estacao, prm_iteracao))

df_saida_termos_init = pd.DataFrame (saida_termos, columns= ['TIMESTAMP', 'IA',\
                                                             'IE_1', 'IE_2', 'IE_3', 'IE_4',\
                                                             'NPAE',\
                                                             'CA_1', 'CA_2', 'CA_3', 'CA_4',\
                                                             'AN',\
                                                             'AW',\
                                                             'FLOP'])
print (df_saida_termos_init)

# primeiro refinamento
lista_termos = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
saida_termos = []

prm_iteracao = 1 

for prm_ano in range (2007, 2017):
    for prm_estacao in range (0,4):
        if (prm_ano == 1999 and prm_estacao == MarchEq): 
            pass
        elif (prm_ano == 2000 and prm_estacao == SeptEq):
            pass
        elif (prm_ano == 2005 and prm_estacao == SeptEq):
            pass
        elif (prm_ano == 2005 and prm_estacao == DecSol):
            pass
        elif (prm_ano == 2016 and prm_estacao == DecSol):
            pass
        elif (prm_ano == 2019 and prm_estacao == DecSol):
            pass
        else:
            saida_termos.append (Le_termos_outmex_file (prm_ano, prm_estacao, prm_iteracao))

df_saida_termos_refina1 = pd.DataFrame (saida_termos, columns= ['TIMESTAMP', 'IA',\
                                                                'IE_1', 'IE_2', 'IE_3', 'IE_4',\
                                                                'NPAE',\
                                                                'CA_1', 'CA_2', 'CA_3', 'CA_4',\
                                                                'AN',\
                                                                'AW',\
                                                                'FLOP'])
print (df_saida_termos_refina1)

df_saida_termos_init.to_excel('TP_term_values_flop0.xls')
df_saida_termos_refina1.to_excel('TP_term_values_flop1.xls')


     TIMESTAMP       IA     IE_1    IE_2    IE_3     IE_4     NPAE     CA_1  \
0   2007-03-20   9.3764   1.7840 -5.1940  0.8607  -1.5540 -20.4098  -0.4836   
1   2007-06-21  -3.2769   4.7746 -5.8871  0.8072  -2.4023   7.7100   1.0095   
2   2007-09-23  -9.3610   6.9482 -2.7998  2.1182  -4.4889 -14.9338  19.8042   
3   2007-12-22 -32.9461   6.6873 -2.6817  1.0675  -2.8972 -34.2403  43.2884   
4   2008-03-20  41.3688   8.0494 -5.1653  0.2134  -2.2510  31.7997 -54.1215   
5   2008-06-21  10.7452   1.2266 -3.5320  1.3905  -0.2668   6.9420 -10.0108   
6   2008-09-23  -5.2249   5.2898 -4.2959 -0.2407  -1.8641  -8.3802  11.8805   
7   2008-12-22   6.8689   7.8510 -4.6126  2.7775  -4.1477   8.0295 -10.4959   
8   2009-03-20  -2.6432   8.4923 -3.9049 -1.7822  -2.5668   4.4296  -2.2098   
9   2009-06-21   1.0561   7.1760 -6.3739  0.5175  -2.5789   1.2957  -0.2178   
10  2009-09-23 -11.2282   5.5603 -4.0039  1.6993  -3.3870 -17.7141  19.2892   
11  2009-12-22 -20.7151   9.1644 -4.3799  2.0636  -4

C:\Users\myrnayk\AppData\Local\Temp\ipykernel_14324\3730284883.py:91: FutureWarning: As the xlwt package is no longer maintained, the xlwt engine will be removed in a future version of pandas. This is the only engine in pandas that supports writing in the xls format. Install openpyxl and write to an xlsx file instead. You can set the option io.excel.xls.writer to 'xlwt' to silence this warning. While this option is deprecated and will also raise a warning, it can be globally set and the warning suppressed.
  df_saida_termos_init.to_excel('TP_term_values_flop0.xls')
C:\Users\myrnayk\AppData\Local\Temp\ipykernel_14324\3730284883.py:92: FutureWarning: As the xlwt package is no longer maintained, the xlwt engine will be removed in a future version of pandas. This is the only engine in pandas that supports writing in the xls format. Install openpyxl and write to an xlsx file instead. You can set the option io.excel.xls.writer to 'xlwt' to silence this warning. While this option is deprecate

### ROTINA PRINCIPAL - FORMATAÇÃO DAS ENTRADAS PARA TPOINT ###
#   29 Jan 2022: 
#       (1) Inclusão do coeficiente FLOP no modelo de apontamento

# NOTA 0: Para usar, mudar a célula de "Markdown" para "Code"
# NOTA 1: Dar shutdown no kernel a cada ano processado para não congelar o computador. 
#         O software requer otimizações.out
# NOTA 2: Loop para estações por fora do loop para canais também 'congela' o Jupyter Notebook.
# NOTA 3: Alterar os valores de ano e de estação e executar esta célula novamente para todos os canais de uma vez.
# NOTA 4: O solstício de dezembro gera o arquivo do verão do ano seguinte.

### ALTERAR A ITERACAO DE REFINAMENTO AQUI
### NÃO É NECESSÁRIO EXECUTAR A ITERAÇÃO 0 PARA NOVOS MODELOS DE APONTAMENTO
iteracao = 1

### ALTERAR O ANO AQUI
ano_ini = 2016

### ALTERAR AQUI O INDICE DA ESTACAO DO ANO
# Retirar o comentário de uma linha por vez para escolher o marco de inicio da estacao desejada
#estacao = MarchEq
#estacao = JuneSol
#estacao = SeptEq
estacao = DecSol

### LEITURA DE MODELO
(tp_IA, tp_IE, tp_NPAE, tp_CA, tp_AN, tp_AW, tp_FLOP) = Le_Modelo_outmex_FLOP (ano_ini, estacao, iteracao)
# echo print
#if (estacao == DecSol):
#    print ('Modelo TPoint:', seasons[estacao], ano_ini+1, '- iteracao:', iteracao)
#else:
#    print ('Modelo TPoint:', seasons[estacao], ano_ini, '- iteracao:', iteracao)
    
### Para cada canal ###
for canal_analise in range (1, 5):
    # zerar os dataframes
    VARGLOBAL_df_entradas_TPoint = pd.DataFrame()
    VARGLOBAL_data_ch_of_interest = pd.DataFrame()

    # Definicao de strings de nome e cabecalho de arquivo, 
    nome_arq = NomeArq_Estacao_Canal (ano_ini, estacao, canal_analise, iteracao)
    periodo_analise = CabArq_Estacao_Canal (ano_ini, estacao, iteracao)

    # echo print
    print ('Processando', nome_arq)

    # Leitura e filtragem dos dados, registro em VARGLOBAL_data_ch_of_interest 
    VARGLOBAL_data_ch_of_interest = Ler_Dados_Estacao_Canal (ano_ini, estacao, canal_analise)

    # Calcular dados de entrada do TPoint de VARGLOBAL_data_ch_of_interest
    # e geracao de VARGLOBAL_df_entradas_TPoint 
    VARGLOBAL_df_entradas_TPoint = Calcula_Dados_TPoint_Estacao_Canal (VARGLOBAL_data_ch_of_interest)

    # Gravacao de VARGLOBAL_df_entradas_TPoint no arquivo de entrada para o TPoint
    Grava_Dados_TPoint_Estacao_Canal (nome_arq, periodo_analise, canal_analise, iteracao, \
                                      VARGLOBAL_df_entradas_TPoint)

# echo print
print ('*** Arquivos gerados - fim do processamento ***')
